In [0]:
import os
from pyspark.sql.functions import col, when, hour, month

os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/workspace/default/mama/sparkml_cache"

df = spark.table("workspace.default.us_accidents_hpms_final_100_m_features")

print(df.count(), len(df.columns))
display(df.limit(5))

4484584 50


Severity,Start_Time,Start_Lat,Start_Lng,Distance(mi),City,County,State,Temperature(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Speed(mph),Precipitation(in),Weather_Condition,Sunrise_Sunset,AADT,AADT_COMBINATION,AADT_SINGLE_UNIT,ACCESS_CONTROL,FACILITY_TYPE,F_SYSTEM,SPEED_LIMIT,SURFACE_TYPE,StateID,THROUGH_LANES,URBAN_ID,hpms_distance_m,Hour,Month,DayOfWeek,Is_Weekend,Is_Peak_Hour,Is_Night,Is_Rain,Is_Snow,Is_Fog,Low_Visibility,High_Wind,Has_Precipitation,Truck_AADT,Truck_Ratio,AADT_per_Lane,Is_High_Speed,Is_Controlled_Access,Is_Urban,Very_Close_HPMS,Rain_LowVis,Snow_LowVis,HighSpeed_Urban
2,2022-05-26T18:24:00.000Z,38.794748,-77.14471800000003,5.749,Alexandria,Fairfax County,VA,72.0,83.0,30.03,10.0,6.0,0.0,Cloudy,Day,172980.0,13470.0,3407.0,1.0,2,1.0,55.0,7.0,51,11.0,92242,1.500479455243109,18.0,5.0,3.0,0,1,0,0,0,0,0,0,0,16877.0,0.097566195,15725.454,0,1,1,1,0,0,0
2,2022-03-01T16:00:00.000Z,38.795042,-77.18656999999997,4.807,Springfield,Fairfax County,VA,56.0,35.0,29.91,10.0,8.0,0.0,Fair,Day,null,null,null,null,6,1.0,null,7.0,51,null,92242,0.8331936120027094,16.0,3.0,1.0,0,1,0,0,0,0,0,0,0,0.0,null,null,0,0,1,1,0,0,0
2,2022-08-28T13:00:30.000Z,37.325115,-80.033187,4.323,Roanoke,Roanoke County,VA,86.0,61.0,28.9,10.0,9.0,0.0,Partly Cloudy,Day,null,null,null,null,6,1.0,null,2.0,51,null,75421,1.660423180908143,13.0,8.0,6.0,1,0,0,0,0,0,0,0,0,0.0,null,null,0,0,1,1,0,0,0
2,2022-09-15T17:27:59.000Z,38.939907,-77.384456,1.749,Herndon,Fairfax County,VA,78.0,42.0,29.81,10.0,9.0,0.0,Fair,Day,null,null,null,null,6,3.0,null,null,51,null,92242,2.1592777364387974,17.0,9.0,3.0,0,1,0,0,0,0,0,0,0,0.0,null,null,0,0,1,1,0,0,0
2,2022-08-19T16:19:29.000Z,38.745025,-77.250609,0.029,Springfield,Fairfax,VA,86.0,52.0,29.96,10.0,9.0,0.0,Partly Cloudy,Day,null,null,null,null,6,3.0,null,null,51,null,92242,53.23505290330992,16.0,8.0,4.0,0,1,0,0,0,0,0,0,0,0.0,null,null,0,0,1,0,0,0,0


In [0]:
use_df = df.limit(500000).select(
    col("Distance(mi)").cast("double").alias("Distance_mi"),
    col("Temperature(F)").cast("double").alias("Temperature_F"),
    col("Humidity(%)").cast("double").alias("Humidity_pct"),
    col("Visibility(mi)").cast("double").alias("Visibility_mi"),
    col("Wind_Speed(mph)").cast("double").alias("Wind_Speed_mph"),
    col("Precipitation(in)").cast("double").alias("Precipitation_in"),
    hour(col("Start_Time")).alias("Hour"),
    month(col("Start_Time")).alias("Month"),
    col("AADT").cast("double").alias("AADT"),
    col("SPEED_LIMIT").cast("double").alias("SPEED_LIMIT"),
    col("THROUGH_LANES").cast("double").alias("THROUGH_LANES"),
    col("URBAN_ID").cast("double").alias("URBAN_ID"),
    col("hpms_distance_m").cast("double").alias("hpms_distance_m"),
    when(col("Severity") <= 2, 0).otherwise(1).alias("label")
).dropna()

print("rows ready:", use_df.count())
display(use_df.limit(5))

rows ready: 386010


Distance_mi,Temperature_F,Humidity_pct,Visibility_mi,Wind_Speed_mph,Precipitation_in,Hour,Month,AADT,SPEED_LIMIT,THROUGH_LANES,URBAN_ID,hpms_distance_m,label
5.749,72.0,83.0,10.0,6.0,0.0,18,5,172980.0,55.0,11.0,92242.0,1.500479455243109,0
1.113,37.0,89.0,3.0,12.0,0.2,12,12,70020.0,60.0,4.0,75421.0,0.4524617631343032,0
0.8759999999999999,79.0,56.0,10.0,6.0,0.0,12,5,70020.0,60.0,4.0,75421.0,1.1725322867301835,0
1.245,74.0,70.0,10.0,10.0,0.0,16,9,56098.0,70.0,4.0,37162.0,0.5794107499909252,0
2.335,23.0,88.0,1.0,14.0,0.0,5,11,9559.0,60.0,4.0,99999.0,0.2526683416320272,0


In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator

feature_cols = [
    "Distance_mi",
    "Temperature_F",
    "Humidity_pct",
    "Visibility_mi",
    "Wind_Speed_mph",
    "Precipitation_in",
    "Hour",
    "Month",
    "AADT",
    "SPEED_LIMIT",
    "THROUGH_LANES",
    "URBAN_ID",
    "hpms_distance_m"
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
scaler = StandardScaler(inputCol="features", outputCol="scaled")

lr = LogisticRegression(
    featuresCol="scaled",
    labelCol="label",
    maxIter=20
)

pipe = Pipeline(stages=[assembler, scaler, lr])

param_grid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01, 0.1, 1.0]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0]) \
    .build()

evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    metricName="areaUnderROC"
)

cv = CrossValidator(
    estimator=pipe,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=3,
    seed=42,
    parallelism=1
)

In [0]:
train_df, test_df = use_df.randomSplit([0.8, 0.2], seed=42)

print("train:", train_df.count())
print("test:", test_df.count())

train_df.groupBy("label").count().orderBy("label").show()
test_df.groupBy("label").count().orderBy("label").show()

train: 308874
test: 77136
+-----+------+
|label| count|
+-----+------+
|    0|223610|
|    1| 85264|
+-----+------+

+-----+-----+
|label|count|
+-----+-----+
|    0|55768|
|    1|21368|
+-----+-----+



In [0]:
import mlflow

with mlflow.start_run(run_name="Phase3_LogisticRegression_HPMS_500k"):
    print("training...")

    cv_model = cv.fit(train_df)
    best_lr = cv_model.bestModel.stages[-1]

    print("best regParam:", best_lr.getRegParam())
    print("best elasticNetParam:", best_lr.getElasticNetParam())

    mlflow.log_param("regParam", best_lr.getRegParam())
    mlflow.log_param("elasticNetParam", best_lr.getElasticNetParam())
    mlflow.log_param("model_type", "LogisticRegression")

training...
best regParam: 0.01
best elasticNetParam: 0.0


In [0]:
import pandas as pd
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

best_model = cv_model.bestModel
pred = best_model.transform(test_df)

auc_eval = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

acc_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

f1_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

wp_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

wr_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
)

results_df = pd.DataFrame({
    "Metric": ["AUC", "Accuracy", "F1 Score", "Weighted Precision", "Weighted Recall"],
    "Value": [
        round(auc_eval.evaluate(pred), 4),
        round(acc_eval.evaluate(pred), 4),
        round(f1_eval.evaluate(pred), 4),
        round(wp_eval.evaluate(pred), 4),
        round(wr_eval.evaluate(pred), 4)
    ]
})

print(results_df.to_string(index=False))

            Metric  Value
               AUC 0.7425
          Accuracy 0.7538
          F1 Score 0.7197
Weighted Precision 0.7310
   Weighted Recall 0.7538


In [0]:
pred.groupBy("label", "prediction").count().orderBy("label", "prediction").show()
display(pred.select("label", "prediction", "probability").limit(20))

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    0|       0.0|52051|
|    0|       1.0| 3717|
|    1|       0.0|15275|
|    1|       1.0| 6093|
+-----+----------+-----+



label,prediction,probability
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.7463616401098552"",""0.2536383598901448""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9173364278820322"",""0.08266357211796782""]}"
1,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.8469475694136144"",""0.15305243058638562""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.7755702371263273"",""0.22442976287367267""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9270205424325518"",""0.07297945756744817""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.8913323091148289"",""0.10866769088517114""]}"
1,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.7308839099319037"",""0.26911609006809634""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9147465631683389"",""0.08525343683166109""]}"
0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9140014337810058"",""0.08599856621899415""]}"
1,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.827578671790477"",""0.17242132820952305""]}"


In [0]:
model_path = "/Volumes/workspace/default/mama/models/logistic_regression_hpms_500k"
best_model.write().overwrite().save(model_path)
print("saved to:", model_path)

saved to: /Volumes/workspace/default/mama/models/logistic_regression_hpms_500k
